In [1]:
from pathlib import Path
from PIL import Image, ImageDraw, ImageFont, ImageFilter
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np
import random
import tensorflow as tf

In [2]:
# Get the project directory
PROJECT_DIR = Path.cwd().parents[0]

In [3]:
# Sizes
IMG_SIZE = 64
BATCH_SIZE = 32

In [4]:
# Digits from 1 to 9
DIGITS = list(range(1, 10))

In [5]:
# The paths of the fonts
FONT_PATHS = [
    "/System/Library/Templates/Data/Library/Fonts/Arial Unicode.ttf",
    "/System/Library/Fonts/Helvetica.ttc",
    "/System/Library/Fonts/Supplemental/AppleGothic.ttf",
]

In [ ]:
def generate_digit_image(digit: int) -> tuple[np.ndarray, int]:
    """
    Generates a given digit on the image with random augmentation.

    Args:
        digit: The given digit

    Returns:
        tuple:
            np.ndarray: The image
            int: The label of the image (zero-based index)
    """

    # Create a greyscale image in grey mode
    img = Image.new("L", (IMG_SIZE, IMG_SIZE), 255)
    draw = ImageDraw.Draw(img)

    # Select a random font
    font_path = random.choice(FONT_PATHS)

    # Select random font size
    font_size = random.randint(30, 60)

    try:
        font = ImageFont.truetype(font_path, font_size)
    except:
        font = ImageFont.load_default()

    # Conver the digit to string format
    text = str(digit)

    # Find the width and height of the text bounding box
    left, top, right, bottom = draw.textbbox((0, 0), text, font=font)
    w = right - left
    h = bottom - top

    # Determine the position of the text
    x = (IMG_SIZE - w) // 2 + random.randint(-5, 5)
    y = (IMG_SIZE - h) // 2 + random.randint(-5, 5)

    draw.text((x, y), text, fill='black', font=font)

    # Blur the image
    if random.random() < 0.4:
        img = img.filter(ImageFilter.GaussianBlur(random.uniform(0, 1.5)))

    # Convert the image to an array
    arr = np.array(img).astype(np.float32)

    # Apply noises
    if random.random() < 0.5:
        arr += np.random.normal(0, 20, arr.shape)

    # Clip all pixels of the image between 0 and 255
    arr = np.clip(arr, 0, 255)

    # Convert the image to RGB
    arr = np.stack([arr, arr, arr], axis=-1)

    return arr.astype(np.float32), (digit - 1)

In [7]:
def data_generator():
    """
    Data generator wrapper that yields an RGB image and its label

    Yields:
        tuple:
            np.ndarray: An RGB image that contains an augmented digit
            int: The label of the image (zero-based index)
    """

    while True:
        digit = random.choice(DIGITS)
        img, label = generate_digit_image(digit)
        yield img, label

In [11]:
# Describe what the custom generator will yield
output_signature = (
    tf.TensorSpec(shape=(IMG_SIZE, IMG_SIZE, 3), dtype=tf.float32), # the image
    tf.TensorSpec(shape=(), dtype=tf.int32) # the label
)

# Create the training dataset from the generator
train_ds = tf.data.Dataset.from_generator(
    data_generator,
    output_signature=output_signature
)

# Use prefetching to improve performance
train_ds = train_ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

In [26]:
class_names = train_ds.class_names
print("Output classes:", class_names)

AttributeError: '_PrefetchDataset' object has no attribute 'class_names'

In [13]:
# Create the training dataset from the generator
val_ds = tf.data.Dataset.from_generator(
    data_generator,
    output_signature=output_signature
)

val_ds = val_ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

In [ ]:
# Customize a model from the base MobileNetV2 model
NUM_CLASSES = len(DIGITS)

# Define the input layer
inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))

# Pass the input through MobileNetV2's specific scalar
x = keras.applications.mobilenet_v2.preprocess_input(inputs)

# Define the base model
base_model = keras.applications.MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights="imagenet"
)
base_model.trainable = False    # freezes the model

# Chain the tensor layers
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(64, activation="relu")(x)
x = layers.Dropout(0.3)(x)

# Define the output layer
outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)

# Create our custom model
model = keras.Model(inputs, outputs)

/var/folders/vb/gdmzt8xs5bb2rp7k5ht40ync0000gn/T/ipykernel_16607/1104157728.py:11: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base_model = keras.applications.MobileNetV2(


In [21]:
model.compile(
    optimizer=keras.optimizers.Adam(1e-4),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

# Summary of the model
model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)      │ (None, 64, 64, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ true_divide_1 (TrueDivide)      │ (None, 64, 64, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ subtract_1 (Subtract)           │ (None, 64, 64, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 2, 2, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │        81,984 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 9)              │           585 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,340,553 (8.93 MB)

 Trainable params: 82,569 (322.54 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [22]:
# Create a callback to stop training when the improvement metric has stopped improving
callback = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss', # set the improvement metric to value loss
    patience=3, # the number of epochs to wait
    restore_best_weights=True   # roll back the model to its best performance state
)

In [23]:
history = model.fit(
    train_ds,
    steps_per_epoch=200,
    epochs=20,
    callbacks=[callback]
)

Epoch 1/20
200/200 ━━━━━━━━━━━━━━━━━━━━ 6s 20ms/step - accuracy: 0.4455 - loss: 1.6932
Epoch 2/20
 10/200 ━━━━━━━━━━━━━━━━━━━━ 3s 19ms/step - accuracy: 0.6376 - loss: 1.0511

/Users/neil/Develop/SudoVisionCruncher/.venv/lib/python3.11/site-packages/keras/src/callbacks/early_stopping.py:99: UserWarning: Early stopping conditioned on metric `val_loss` which is not available. Available metrics are: accuracy,loss
  current = self.get_monitor_value(logs)


200/200 ━━━━━━━━━━━━━━━━━━━━ 4s 19ms/step - accuracy: 0.7184 - loss: 0.8020
Epoch 3/20
200/200 ━━━━━━━━━━━━━━━━━━━━ 4s 19ms/step - accuracy: 0.7980 - loss: 0.5602
Epoch 4/20
200/200 ━━━━━━━━━━━━━━━━━━━━ 4s 20ms/step - accuracy: 0.8458 - loss: 0.4328
Epoch 5/20
200/200 ━━━━━━━━━━━━━━━━━━━━ 4s 19ms/step - accuracy: 0.8800 - loss: 0.3568
Epoch 6/20
200/200 ━━━━━━━━━━━━━━━━━━━━ 4s 19ms/step - accuracy: 0.9036 - loss: 0.2961
Epoch 7/20
200/200 ━━━━━━━━━━━━━━━━━━━━ 4s 20ms/step - accuracy: 0.9075 - loss: 0.2675
Epoch 8/20
200/200 ━━━━━━━━━━━━━━━━━━━━ 4s 19ms/step - accuracy: 0.9220 - loss: 0.2358
Epoch 9/20
200/200 ━━━━━━━━━━━━━━━━━━━━ 4s 19ms/step - accuracy: 0.9303 - loss: 0.2109
Epoch 10/20
200/200 ━━━━━━━━━━━━━━━━━━━━ 4s 20ms/step - accuracy: 0.9345 - loss: 0.1955
Epoch 11/20
200/200 ━━━━━━━━━━━━━━━━━━━━ 4s 19ms/step - accuracy: 0.9355 - loss: 0.1878
Epoch 12/20
200/200 ━━━━━━━━━━━━━━━━━━━━ 4s 20ms/step - accuracy: 0.9438 - loss: 0.1651
Epoch 13/20
200/200 ━━━━━━━━━━━━━━━━━━━━ 4s 20ms/st

In [24]:
# Evaluate the model
test_loss, test_acc = model.evaluate(val_ds, steps=5, verbose=2)
print(f"\nTest Accuracy: {test_acc * 100:.2f}%")

5/5 - 1s - 231ms/step - accuracy: 0.9875 - loss: 0.0770

Test Accuracy: 98.75%


In [25]:
# Save the model and its parameters
model.save(f'{PROJECT_DIR}/models/font_recognition_MobileNetV2.keras')